In [ ]:
import os
import pandas as pd
from google.colab import drive

# ===============================================================================
# STEP 1: SETUP AND DRIVE MOUNTING
# ===============================================================================

try:
    drive.mount('/content/drive')
except:
    print("Drive already mounted or error mounting.")

# --- Define the paths for your files ---
# Folder containing the split file and where results will be saved
PROJECT_FOLDER = '/content/drive/My Drive/ProjectFolder/'

In [ ]:
# Folder containing the subdirectories with the essays
ESSAYS_FOLDER = os.path.join(PROJECT_FOLDER, 'brat-project-final')

# Path to the split file
SPLIT_CSV_FILE = os.path.join(PROJECT_FOLDER, 'aae-train-test-split.csv')

In [ ]:
# ===============================================================================
# STEP 2: FUNCTION TO EXTRACT DATA FROM .ANN AND .TXT FILES
# ===============================================================================

def extract_aae_data(essays_dir):
    """
    Traverses the essays folder, extracts arguments, topics, and their relations.

    Args:
        essays_dir (str): Path to the folder containing essayXXX.txt and .ann files.

    Returns:
        pd.DataFrame: A DataFrame with columns 'ID', 'topic', 'argument', and 'classification'.
    """
    dataset = []

    # Validate if the essays directory exists
    if not os.path.exists(essays_dir):
        print(f"ERROR: Essays directory not found at '{essays_dir}'")
        return pd.DataFrame()

    print(f"Starting extraction of essays from '{essays_dir}'...")

    # Iterate over essay numbers from 1 to 402
    for i in range(1, 403):
        # Format the base file name, e.g., "essay001", "essay023", "essay402"
        base_name = f"essay{i:03d}"

        ann_file = os.path.join(essays_dir, f"{base_name}.ann")
        txt_file = os.path.join(essays_dir, f"{base_name}.txt")

        if os.path.exists(ann_file) and os.path.exists(txt_file):
            try:
                # Dictionaries to map component IDs to their texts and types
                components = {} # Maps T_id -> (type, text)

                # Read the .ann file content to extract components first
                with open(ann_file, 'r', encoding='utf-8') as f_ann:
                    for line in f_ann:
                        line = line.strip()
                        if line.startswith('T'): # Text component line (T1, T2, etc.)
                            parts = line.split('\t')
                            component_id = parts[0]
                            type_and_indices = parts[1].split()
                            comp_type = type_and_indices[0] # E.g., "Claim", "Premise"
                            text = parts[2]
                            components[component_id] = {'type': comp_type, 'text': text}

                # Now, process the relations in the same .ann file
                with open(ann_file, 'r', encoding='utf-8') as f_ann:
                    for line in f_ann:
                        line = line.strip()
                        if line.startswith('R'): # Relation line (R1, R2, etc.)
                            parts = line.split('\t')[1].split()
                            relation_type = parts[0] # "supports" or "attacks"

                            # Extract the IDs of the argument and the target (claim)
                            id_arg = parts[1].split(':')[1] # Arg1 is always the Premise (argument)
                            id_target = parts[2].split(':')[1] # Arg2 is always the Claim (topic)

                            # The logic here is that a Premise (argument) supports/attacks a Claim (topic)
                            if components.get(id_arg) and components.get(id_target):
                                argument_text = components[id_arg]['text']
                                topic_text = components[id_target]['text']

                                classification = 'SUPPORT' if relation_type == 'supports' else 'ATTACK'

                                dataset.append({
                                    'ID': f"{base_name}", # Add the ID column
                                    'topic': topic_text,
                                    'argument': argument_text,
                                    'classification': classification
                                })
            except Exception as e:
                print(f"Error processing file {base_name}: {e}")

    print("Extraction completed.")
    return pd.DataFrame(dataset)

In [ ]:
# ===============================================================================
# STEP 3: EXECUTION, MERGE, AND SAVING
# ===============================================================================

# 1. Extract the raw data from the essays
df_extracted = extract_aae_data(ESSAYS_FOLDER)

if not df_extracted.empty:
    print(f"\n{len(df_extracted)} argument relations were extracted.")
    display(df_extracted.head())

    # 2. Load the split file (train/test)
    try:
        df_split = pd.read_csv(SPLIT_CSV_FILE, sep=';')
        print(f"\nSplit file '{os.path.basename(SPLIT_CSV_FILE)}' loaded successfully.")
        display(df_split.head())

        # 3. Merge the two DataFrames using the 'ID' column
        df_complete = pd.merge(df_extracted, df_split, on='ID', how='left')
        print("\nMerge completed successfully.")

        # Reorder columns for better visualization
        if 'SET' in df_complete.columns:
            df_complete = df_complete[['ID', 'SET', 'topic', 'argument', 'classification']]

        # 4. Save the complete dataset
        complete_path = os.path.join(PROJECT_FOLDER, 'aae_complete.csv')
        df_complete.to_csv(complete_path, index=False, encoding='utf-8')
        print(f"\n✅ Complete dataset saved at: '{complete_path}'")
        display(df_complete.head())

        # 5. Filter and save the test dataset
        if 'SET' in df_complete.columns:
            df_test = df_complete[df_complete['SET'] == 'TEST'].copy()
            test_path = os.path.join(PROJECT_FOLDER, 'aae_test.csv')
            df_test.to_csv(test_path, index=False, encoding='utf-8')
            print(f"\n✅ Test dataset ({len(df_test)} rows) saved at: '{test_path}'")
            display(df_test.head())
        else:
            print("\nWARNING: The 'SET' column was not found after the merge. The test file was not generated.")

    except FileNotFoundError:
        print(f"\n❌ ERROR: Split file not found at '{SPLIT_CSV_FILE}'. Merge could not be performed.")
    except Exception as e:
        print(f"\nAn error occurred during merge or saving: {e}")